In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

# Assuming TSB_AD is installed correctly and import paths are relative if running from within the repo
# Or adjust imports if TSB_AD is installed as a package
try:
    # If running from within the TSB-AD repo structure
    from TSB_AD.model_wrapper import (
        run_Unsupervise_AD, Unsupervise_AD_Pool,
        run_Semisupervise_AD, Semisupervise_AD_Pool # <-- Import Semi-supervised components
    )
    from TSB_AD.evaluation.metrics import get_metrics
    # Import only the existing HP dictionary
    from TSB_AD.HP_list import Optimal_Uni_algo_HP_dict # <-- Only import this one
    from TSB_AD.utils.slidingWindows import find_length_rank
except ImportError:
    # If TSB_AD is installed as a package (adjust if needed based on your setup)
    print("Attempting package import style")
    from TSB_AD.model_wrapper import (
        run_Unsupervise_AD, Unsupervise_AD_Pool,
        run_Semisupervise_AD, Semisupervise_AD_Pool # <-- Import Semi-supervised components
    )
    from TSB_AD.evaluation.metrics import get_metrics
     # Import only the existing HP dictionary
    from TSB_AD.HP_list import Optimal_Uni_algo_HP_dict # <-- Only import this one
    from TSB_AD.utils.slidingWindows import find_length_rank


# Specify Anomaly Detector to use and data directory
AD_Name = 'CNN'
data_path = '/Users/kai/Documents/Time_Series_Anomaly_Detection_Seminar/Time-Series-Anomaly-Detection-Seminar/TSB-AD/Datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv'
filename = os.path.basename(data_path)

# Loading Data
df = pd.read_csv(data_path).dropna()
data = df.iloc[:, 0:-1].values.astype(float)
label = df['Label'].astype(int).to_numpy()

# --- Define data_train (Needed for Semi-supervised models) ---
try:
    train_index_str = filename.split('.')[0].split('_')[-3]
    if train_index_str == 'tr' and filename.split('.')[0].split('_')[-2].isdigit():
         train_index = int(filename.split('.')[0].split('_')[-2])
         print(f"Using train_index derived from filename: {train_index}")
    elif train_index_str.isdigit():
         train_index = int(train_index_str)
         print(f"Using train_index derived from filename (fallback position): {train_index}")
    else:
        raise ValueError("Filename does not match expected 'tr_INDEX' pattern.")

    if train_index <= 0 or train_index >= len(data):
         raise ValueError(f"Derived train_index {train_index} is out of bounds for data length {len(data)}.")
    data_train = data[:train_index, :]
except (IndexError, ValueError) as e:
    print(f"Warning: Could not derive train_index from filename '{filename}' ({e}). Falling back to 50% split.")
    split_point = int(len(data) * 0.5)
    if split_point == 0 and len(data) > 0:
        split_point = 1
    print(f"Using fallback train split: first {split_point} points.")
    data_train = data[:split_point, :]

if data_train.shape[0] == 0:
     raise ValueError("Training data segment (data_train) is empty. Check split logic or data length.")
print(f"Data shape: {data.shape}, Training data shape: {data_train.shape}")

# --- Get Hyperparameters and Run Model ---
# Fetch HPs from the single consolidated dictionary
if AD_Name in Optimal_Uni_algo_HP_dict:
    Optimal_Det_HP = Optimal_Uni_algo_HP_dict[AD_Name]
    print(f"Using Optimal Hyperparameters for {AD_Name} from Optimal_Uni_algo_HP_dict: {Optimal_Det_HP}")
else:
    print(f"Warning: No optimal hyperparameters found for {AD_Name} in Optimal_Uni_algo_HP_dict. Using default parameters.")
    Optimal_Det_HP = {} # Pass an empty dict if none are found

output = None
print(f"Running {AD_Name}...")
try:
    # Still need to choose the correct WRAPPER function based on the pool
    if AD_Name in Semisupervise_AD_Pool:
        print("Using run_Semisupervise_AD wrapper.")
        # Pass HPs obtained from Optimal_Uni_algo_HP_dict
        output = run_Semisupervise_AD(AD_Name, data_train, data, **Optimal_Det_HP)
    elif AD_Name in Unsupervise_AD_Pool:
        print("Using run_Unsupervise_AD wrapper.")
         # Pass HPs obtained from Optimal_Uni_algo_HP_dict
        output = run_Unsupervise_AD(AD_Name, data, **Optimal_Det_HP)
    else:
        # Check the pools defined in model_wrapper.py if unsure where a model is categorized
        available_models = list(Unsupervise_AD_Pool) + list(Semisupervise_AD_Pool)
        raise ValueError(f"'{AD_Name}' is not defined in Semisupervise_AD_Pool or Unsupervise_AD_Pool. Available models in pools: {available_models}")

except Exception as e:
    print(f"An error occurred during model execution: {e}")
    # Optional: Re-raise to see the full traceback for debugging
    # import traceback
    # traceback.print_exc()
    raise e


# --- Evaluation (remains the same) ---
slidingWindow = find_length_rank(data, rank=1)
print(f"Calculated sliding window: {slidingWindow}")

if isinstance(output, np.ndarray):
    print(f"{AD_Name} executed successfully. Output shape: {output.shape}")

    if len(output) != len(label):
         print(f"Warning: Output length ({len(output)}) does not match label length ({len(label)}). Evaluation might be incorrect.")
         # Adjust length if necessary and appropriate for the specific model's behavior
         # output = output[:len(label)] # Example: Simple truncation

    # Scale the scores to [0, 1] range
    if output.ndim == 1:
        output_scaled = MinMaxScaler(feature_range=(0,1)).fit_transform(output.reshape(-1,1)).ravel()
    elif output.ndim == 2 and output.shape[1] == 1:
         output_scaled = MinMaxScaler(feature_range=(0,1)).fit_transform(output).ravel()
    else:
        print(f"Warning: Output shape {output.shape} might not be compatible with MinMaxScaler. Trying to flatten.")
        try:
            output_scaled = MinMaxScaler(feature_range=(0,1)).fit_transform(output.reshape(-1,1)).ravel()
            if len(output_scaled) != len(label):
                 raise ValueError("Scaled output length doesn't match label length after reshaping.")
        except Exception as scale_e:
             print(f"Error during scaling: {scale_e}. Skipping scaling.")
             output_scaled = output

    # Final check before evaluation
    if len(output_scaled) == len(label):
        threshold = np.mean(output_scaled) + 3 * np.std(output_scaled)
        threshold = np.clip(threshold, 0, 1)
        pred = (output_scaled > threshold).astype(int)
        print(f"Using threshold {threshold:.4f} for binary predictions.")

        print("Calculating evaluation metrics...")
        evaluation_result = get_metrics(score=output_scaled, labels=label, slidingWindow=slidingWindow, pred=pred)

        print('\nEvaluation Result:')
        for metric, value in evaluation_result.items():
            print(f"  {metric}: {value:.4f}")
    else:
         print(f"ERROR: Final score length ({len(output_scaled)}) does not match label length ({len(label)}). Aborting evaluation.")


elif output is not None:
    print(f'Execution failed or returned non-array output: {output}')
else:
    print("Execution did not produce an output array. Check previous errors.")

Using train_index derived from filename (fallback position): 1007
Data shape: (4031, 1), Training data shape: (1007, 1)
Using Optimal Hyperparameters for CNN from Optimal_Uni_algo_HP_dict: {'window_size': 50, 'num_channel': [32, 32, 40]}
Running CNN...
Using run_Semisupervise_AD wrapper.
----- GPU is unavailable -----
----- Using CPU -----


/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Validation Epoch [3/50]: 100%|██████████| 2/2 [00:00<00:00, 241.34it/s, avg_loss=0.919, loss=0.873]


EarlyStopping counter: 1 out of 3


Validation Epoch [6/50]: 100%|██████████| 2/2 [00:00<00:00, 217.20it/s, avg_loss=0.914, loss=0.868]


EarlyStopping counter: 1 out of 3


...

EarlyStopping counter: 2 out of 3


Validation Epoch [8/50]: 100%|██████████| 2/2 [00:00<00:00, 218.22it/s, avg_loss=0.913, loss=0.871]


EarlyStopping counter: 3 out of 3
torch.Size([]) torch.Size([])
   Early stopping<<<


Testing: : 100%|██████████| 32/32 [00:00<00:00, 169.11it/s]


scores:  (3981,)
Calculated sliding window: 6
CNN executed successfully. Output shape: (4031,)
Using threshold 0.0523 for binary predictions.
Calculating evaluation metrics...

Evaluation Result:
  AUC-PR: 0.1461
  AUC-ROC: 0.5271
  VUS-PR: 0.1425
  VUS-ROC: 0.5317
  Standard-F1: 0.0567
  PA-F1: 1.0000
  Event-based-F1: 1.0000
  R-based-F1: 0.3529
  Affiliation-F: 0.9526
